# Stochastic Differential Equations: Ion Channel Dynamics

## Overview
At small scales (few ion channels), stochastic fluctuations become important. Stochastic Differential Equations (SDEs) extend ODEs by adding random noise.

## Mathematical Framework

### General SDE Form:
$$dX_t = f(X_t, t) dt + g(X_t, t) dW_t$$

where:
- $f(X_t, t)$: deterministic drift term (ODE part)
- $g(X_t, t)$: diffusion coefficient (noise intensity)
- $dW_t$: Wiener process (Brownian motion)

### Ion Channel Kinetics:
For a population of $N_{\text{total}}$ K⁺ channels:

$$dN_{\text{open}} = \left[\alpha(V)(N_{\text{total}} - N_{\text{open}}) - \beta(V)N_{\text{open}}\right]dt + \sqrt{\alpha(V)(N_{\text{total}}-N_{\text{open}}) + \beta(V)N_{\text{open}}} \, dW_t$$

where:
- $\alpha(V)$: voltage-dependent opening rate
- $\beta(V)$: voltage-dependent closing rate
- $N_{\text{open}}$: number of open channels

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)
np.random.seed(42)  # For reproducibility

## Model Parameters

In [ ]:
# Parameters
N_total = 100        # Total number of channels
V_rest = -65.0       # Resting potential (mV)
V_depol = 0.0        # Depolarized potential (mV)
dt = 0.01            # Time step (ms)
T = 100.0            # Total simulation time (ms)
n_steps = int(T / dt)

print(f"Simulation parameters:")
print(f"  Total channels: {N_total}")
print(f"  Time step: {dt} ms")
print(f"  Duration: {T} ms")
print(f"  Number of steps: {n_steps}")

## Voltage-Dependent Rate Functions
Simplified K⁺ channel kinetics (from HH model)

In [ ]:
def alpha(V):
    """Opening rate (1/ms)"""
    return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))

def beta(V):
    """Closing rate (1/ms)"""
    return 0.125 * np.exp(-(V + 65.0) / 80.0)

# Visualize rate functions
V_range = np.linspace(-100, 50, 1000)
alpha_vals = [alpha(v) for v in V_range]
beta_vals = [beta(v) for v in V_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Rates
ax1.plot(V_range, alpha_vals, 'g', linewidth=2, label='α(V) - opening')
ax1.plot(V_range, beta_vals, 'r', linewidth=2, label='β(V) - closing')
ax1.set_xlabel('Voltage (mV)', fontsize=12)
ax1.set_ylabel('Rate (1/ms)', fontsize=12)
ax1.set_title('Voltage-Dependent Rate Functions', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axvline(V_rest, color='b', linestyle='--', alpha=0.5, label=f'Rest ({V_rest} mV)')
ax1.axvline(V_depol, color='orange', linestyle='--', alpha=0.5, label=f'Depol ({V_depol} mV)')

# Steady-state open probability
p_inf = [alpha(v) / (alpha(v) + beta(v)) for v in V_range]
ax2.plot(V_range, p_inf, 'b', linewidth=2)
ax2.set_xlabel('Voltage (mV)', fontsize=12)
ax2.set_ylabel('Open Probability', fontsize=12)
ax2.set_title('Steady-State Open Probability', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axvline(V_rest, color='b', linestyle='--', alpha=0.5)
ax2.axvline(V_depol, color='orange', linestyle='--', alpha=0.5)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('outputs/sde_rate_functions.png', dpi=150, bbox_inches='tight')
plt.show()

## Voltage Stimulus Protocol
Step depolarization to activate channels

In [ ]:
def V_stim(t, t_start=20.0, t_end=70.0):
    """Voltage clamp protocol"""
    if t_start <= t <= t_end:
        return V_depol  # Depolarization step
    else:
        return V_rest   # Resting

# Visualize stimulus
t_test = np.linspace(0, T, 1000)
V_test = [V_stim(t) for t in t_test]

plt.figure(figsize=(12, 4))
plt.plot(t_test, V_test, 'k', linewidth=2)
plt.xlabel('Time (ms)', fontsize=12)
plt.ylabel('Voltage (mV)', fontsize=12)
plt.title('Voltage Clamp Protocol', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/sde_stimulus.png', dpi=150, bbox_inches='tight')
plt.show()

## Euler-Maruyama Method for SDEs

The Euler-Maruyama method extends Euler's method to SDEs:

$$X_{n+1} = X_n + f(X_n, t_n)\Delta t + g(X_n, t_n)\Delta W_n$$

where $\Delta W_n \sim \mathcal{N}(0, \sqrt{\Delta t})$

In [ ]:
def simulate_stochastic_channels(N_total, dt, n_steps, seed=None):
    """
    Simulate stochastic ion channel dynamics using Euler-Maruyama method
    
    SDE: dN = [α(V)(N_total - N) - β(V)N]dt + sqrt[α(V)(N_total-N) + β(V)N] dW
    """
    if seed is not None:
        np.random.seed(seed)
    
    t = np.zeros(n_steps)
    N_open = np.zeros(n_steps)
    V = np.zeros(n_steps)
    
    # Initial condition (equilibrium at rest)
    V[0] = V_rest
    N_open[0] = N_total * alpha(V[0]) / (alpha(V[0]) + beta(V[0]))
    
    for i in range(1, n_steps):
        t[i] = i * dt
        V[i] = V_stim(t[i])
        
        # Current state
        N_o = N_open[i-1]
        N_c = N_total - N_o  # Closed channels
        
        a = alpha(V[i])
        b = beta(V[i])
        
        # Drift term (deterministic)
        drift = a * N_c - b * N_o
        
        # Diffusion term (noise intensity)
        diffusion = np.sqrt(a * N_c + b * N_o)
        
        # Wiener increment: ΔW ~ N(0, sqrt(dt))
        dW = np.random.normal(0, np.sqrt(dt))
        
        # Euler-Maruyama update
        N_open[i] = N_o + drift * dt + diffusion * dW
        
        # Enforce bounds [0, N_total]
        N_open[i] = np.clip(N_open[i], 0, N_total)
    
    return t, V, N_open

## Run Multiple Stochastic Trials
Demonstrate inherent variability

In [ ]:
# Run multiple trials
n_trials = 10
trials_data = []

print(f"Running {n_trials} stochastic trials...")
for trial in range(n_trials):
    t, V, N_open = simulate_stochastic_channels(N_total, dt, n_steps, seed=trial)
    trials_data.append((t, V, N_open))
print("Simulations complete!")

## Visualization: Stochastic Variability

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# Plot 1: Voltage protocol
axes[0].plot(trials_data[0][0], trials_data[0][1], 'k', linewidth=2)
axes[0].set_ylabel('Voltage (mV)', fontsize=12)
axes[0].set_title('Stochastic Ion Channel Dynamics', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Individual trials
colors = plt.cm.viridis(np.linspace(0, 1, n_trials))
for i, (t, V, N_open) in enumerate(trials_data):
    axes[1].plot(t, N_open, color=colors[i], alpha=0.6, linewidth=1.5)

axes[1].set_ylabel('Open Channels', fontsize=12)
axes[1].set_title(f'{n_trials} Stochastic Realizations', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(N_total, color='r', linestyle='--', alpha=0.5, label=f'Max ({N_total})')
axes[1].legend(fontsize=10)

# Plot 3: Mean and variability
N_open_array = np.array([trial[2] for trial in trials_data])
mean_N = N_open_array.mean(axis=0)
std_N = N_open_array.std(axis=0)

axes[2].plot(t, mean_N, 'b', linewidth=3, label='Mean')
axes[2].fill_between(t, mean_N - std_N, mean_N + std_N, 
                      color='b', alpha=0.3, label='± 1 SD')
axes[2].set_xlabel('Time (ms)', fontsize=12)
axes[2].set_ylabel('Open Channels', fontsize=12)
axes[2].set_title('Mean ± Standard Deviation', fontsize=13)
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=11)

plt.tight_layout()
plt.savefig('outputs/sde_stochastic_trials.png', dpi=150, bbox_inches='tight')
plt.show()

## Compare Deterministic vs Stochastic

In [ ]:
# Deterministic solution (no noise)
def simulate_deterministic(N_total, dt, n_steps):
    """Deterministic ODE solution (no stochastic term)"""
    t = np.zeros(n_steps)
    N_open = np.zeros(n_steps)
    V = np.zeros(n_steps)
    
    V[0] = V_rest
    N_open[0] = N_total * alpha(V[0]) / (alpha(V[0]) + beta(V[0]))
    
    for i in range(1, n_steps):
        t[i] = i * dt
        V[i] = V_stim(t[i])
        
        N_o = N_open[i-1]
        N_c = N_total - N_o
        
        a = alpha(V[i])
        b = beta(V[i])
        
        # Only drift, no diffusion
        drift = a * N_c - b * N_o
        N_open[i] = N_o + drift * dt
        N_open[i] = np.clip(N_open[i], 0, N_total)
    
    return t, V, N_open

# Run deterministic
t_det, V_det, N_det = simulate_deterministic(N_total, dt, n_steps)

# Compare
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Stochastic trials (faint)
for i, (t, V, N_open) in enumerate(trials_data[:5]):
    ax.plot(t, N_open, color='gray', alpha=0.3, linewidth=1)

# Mean of stochastic
ax.plot(t, mean_N, 'b', linewidth=2.5, label='Stochastic (mean)', alpha=0.8)

# Deterministic
ax.plot(t_det, N_det, 'r--', linewidth=3, label='Deterministic (ODE)')

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Open Channels', fontsize=12)
ax.set_title('Deterministic vs Stochastic Dynamics', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/sde_deterministic_vs_stochastic.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observation:")
print("  - Deterministic (red) is smooth")
print("  - Stochastic (gray) shows intrinsic fluctuations")
print("  - Mean of stochastic (blue) approximates deterministic")

## Effect of Channel Number
Stochasticity decreases with increasing N (Law of Large Numbers)

In [ ]:
N_values = [10, 50, 100, 500]

fig, axes = plt.subplots(len(N_values), 1, figsize=(14, 12), sharex=True)

for idx, N in enumerate(N_values):
    # Run 3 trials for each N
    for trial in range(3):
        t, V, N_open = simulate_stochastic_channels(N, dt, n_steps, seed=trial)
        # Convert to open probability
        p_open = N_open / N
        axes[idx].plot(t, p_open, alpha=0.7, linewidth=1.5)
    
    axes[idx].set_ylabel('P(open)', fontsize=11)
    axes[idx].set_title(f'N = {N} channels', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0, 1])

axes[-1].set_xlabel('Time (ms)', fontsize=12)
fig.suptitle('Channel Noise Decreases with Population Size', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/sde_channel_number.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nObservation:")
print("  As N increases, fluctuations (noise) decrease")
print("  This is the Law of Large Numbers in action!")

## Statistical Analysis
Quantify the noise level (coefficient of variation)

In [ ]:
N_test = [10, 25, 50, 100, 200, 500, 1000]
cv_values = []  # Coefficient of variation

for N in N_test:
    trials = []
    for trial in range(20):  # 20 trials per N
        _, _, N_open = simulate_stochastic_channels(N, dt, n_steps, seed=trial)
        # Focus on steady-state (50-70 ms)
        idx_start = int(50 / dt)
        idx_end = int(70 / dt)
        mean_steady = N_open[idx_start:idx_end].mean()
        trials.append(mean_steady / N)  # Normalize
    
    # Coefficient of variation: std / mean
    cv = np.std(trials) / np.mean(trials)
    cv_values.append(cv)

# Theoretical: CV ∝ 1/sqrt(N)
cv_theory = [cv_values[0] * np.sqrt(N_test[0] / N) for N in N_test]

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.loglog(N_test, cv_values, 'bo-', linewidth=2, markersize=8, label='Simulated')
ax.loglog(N_test, cv_theory, 'r--', linewidth=2, label='Theory (∝ 1/√N)')
ax.set_xlabel('Number of Channels (N)', fontsize=12)
ax.set_ylabel('Coefficient of Variation (CV)', fontsize=12)
ax.set_title('Noise Scales as 1/√N', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('outputs/sde_noise_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nNoise scaling law:")
print("  CV ∝ 1/√N")
print("  Doubling N reduces noise by ~30%")
print("  10x increase in N reduces noise by ~68%")

## Key Takeaways

1. **SDEs = ODEs + Noise**: Add stochastic term to deterministic dynamics
2. **Euler-Maruyama**: Simple numerical method for SDEs
3. **Biological Relevance**: Important at small scales (few channels, synapses)
4. **Law of Large Numbers**: Noise decreases as √N
5. **Deterministic Limit**: Large populations behave deterministically

## When to Use SDEs

- Small populations (< 100 molecules/channels)
- Single neuron spontaneous activity
- Synaptic transmission variability
- Gene expression dynamics
- Molecular signaling cascades

## Further Exploration

- Implement other integration methods (Milstein, Runge-Kutta for SDEs)
- Add voltage dynamics (stochastic HH neuron)
- Explore different noise sources (additive vs multiplicative)
- Study first-passage time problems